In [22]:
import numpy as np
import yaml

In [23]:
# open input.yaml
with open("input.yaml", "r") as f:
    input_file = yaml.safe_load(f)

# automatic generation of many displacement-controlled steps
# n_steps = input_file["n_steps"]
# print(f"n_steps from input.yaml = {n_steps}")

delta_u1 = 1e-8
u1 = 5e-6

n_steps = int(u1 / delta_u1)
u_vals = [float(i * delta_u1) for i in range(n_steps + 1)]

class IndentedList(list): pass
steps = [IndentedList([0.0, u]) for u in u_vals]

print(f"steps generated = {len(steps)}")

steps generated = 501


In [24]:
import yaml

def indented_list_presenter(dumper, data):
    return dumper.represent_sequence('tag:yaml.org,2002:seq', data, flow_style=True)

yaml.add_representer(IndentedList, indented_list_presenter)

def generate_z3st_bc(filename="boundary_conditions.yaml"):
    # Structure
    bc_data = {
        "mechanical": {
            "steel": [
                {"type": "Clamp_x", "region": "xmax"},
                {"type": "Clamp_y", "region": "ymin"},
                {
                    "region": "ymax",
                    "type": "Dirichlet",
                    "displacement": steps
                }
            ]
        },
        "thermal": {
            "steel": [
                {
                    "type": "Dirichlet", 
                    "region": "xmax", 
                    "temperature": 300.0
                }
            ]
        },
        "damage": {
            "steel": [
                {
                    "type": "Dirichlet", 
                    "region": "crack", 
                    "value": 1.0
                }
            ]
        }
    }

    # Writing
    with open(filename, 'w') as file:
        yaml.dump(bc_data, file, sort_keys=False, default_flow_style=False, indent=2)
    
    print(f"'{filename}' generated.")

if __name__ == "__main__":
    generate_z3st_bc()

'boundary_conditions.yaml' generated.
